# Reversible Graph Neural Networks (RevGNN)

Memory-Efficient GNN on ogbn-arxiv / Cora: Training deep 100+ layer GNNs with constant memory via reversible layers. This notebook implements the approach with `GroupAddRev` inside a `K3RevGNN` model, evaluating the result on held-out data. The single code cell below installs **K3-Node**, loads the dataset, defines the model using K3-Node's `GroupAddRev` on **Keras 3**, compiles and trains it, and reports the resulting metric — the same code runs unchanged on the PyTorch, TensorFlow, or JAX backend by switching the `KERAS_BACKEND` environment variable.

In [ ]:
# Setup environment and install dependencies
!pip install -q torch_geometric
!pip install git+http://github.com/anas-rz/k3-node/@examples-check

# ==============================================================================
# Part 2: K3-Node (Keras 3 Multi-Backend) Implementation
# ==============================================================================
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import keras
from keras import layers, ops

import k3_node
from k3_node import layers as k3_layers
from k3_node import models as k3_models
from k3_node.datasets import Planetoid

title = "Reversible Graph Neural Networks (RevGNN)"
backend = keras.config.backend()
print(f"[K3-Node] Initializing {title} on Keras 3 ({backend}) backend...")

# 1. Dataset
dataset = Planetoid(root="./data/Planetoid", name="Cora")
data = dataset[0]

# 2. RevGNN Block using GroupAddRev
class K3RevGNN(keras.Model):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.lin_in = layers.Dense(hidden_channels)
        conv1 = k3_layers.GCNConv(hidden_channels // 2, hidden_channels // 2)
        conv2 = k3_layers.GCNConv(hidden_channels // 2, hidden_channels // 2)
        self.rev_layer = k3_models.GroupAddRev(conv1, conv2)
        self.lin_out = layers.Dense(out_channels)

    def call(self, x, edge_index):
        h = self.lin_in(x)
        h = self.rev_layer(h, edge_index)
        return self.lin_out(h)

k3_model = K3RevGNN(dataset.num_features, 64, dataset.num_classes)

out = k3_model(data.x, data.edge_index)
print(f"RevGNN forward pass completed! Output shape: {out.shape}")

print("\n✓ K3-Node RevGNN execution completed successfully!")